# Training

Reproducible Ultralytics training entry point. Tune parameters only after validating and preparing the dataset.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ultralytics import YOLO
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.device import select_device

config = load_config(ROOT / 'configs' / 'project.yaml')
device = select_device()
logger = configure_logger(ROOT / config['project']['logs_dir'], 'training')
data_yaml = ROOT / config['project']['processed_data_dir'] / 'yolo_v1' / 'data.yaml'
model = YOLO('yolo11n.pt')
results = model.train(data=str(data_yaml), epochs=100, imgsz=640, batch=16, device=device.name, seed=config['project']['seed'], deterministic=True, project=str(ROOT / config['project']['runs_dir']), name=f'train_{utc_timestamp()}', exist_ok=False)
metadata = {'device': device.__dict__, 'data_yaml': str(data_yaml), 'results': str(results)}
write_json(ROOT / config['project']['reports_dir'] / f'training_{utc_timestamp()}.json', metadata)
logger.info('Training completed on %s', device.name)
print(metadata)